In [1]:
# suppress tensorflow logging, usually not useful unless you are having problems with tensorflow or accessing gpu
# it seems necessary to have this environment variable set before tensorflow is imported, or else it doesn't take effect
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

# imports generally useful throughout the notebook
# usually all imports should happen at the top of a notebook, but in
# these notebooks where the purpose is to show how to use the Keras API
# the relevant imports will happen in the cells where the API is discussed
import pathlib
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# global settings for notebook output and images
plt.rcParams['figure.figsize'] = (8, 8) # set default figure size, 10in by 8in
np.set_printoptions(precision=4, suppress=True)

E0000 00:00:1750182849.565790   42237 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750182849.570484   42237 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750182849.584765   42237 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750182849.584792   42237 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750182849.584794   42237 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750182849.584796   42237 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
# import project defined modules / functions used in this notebook
# ensure that the src directory where project modules are found is on
# the PYTHONPATH
import sys
sys.path.append("../src")

# assignment function imports for doctests and github autograding
# these are required for assignment autograding
from nndl import vectorize_samples, plot_history

In [3]:
# if want to restrict to cpu or gpu, configure visible device for rest of notebook to use
dev = tf.config.list_physical_devices()
print('Physical Devices : ', dev)

#tf.config.set_visible_devices(dev[0])
#tf.config.set_visible_devices(dev[1])
dev = tf.config.list_logical_devices()
print('Available Devices : ', dev)

Physical Devices :  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Available Devices :  [LogicalDevice(name='/device:CPU:0', device_type='CPU'), LogicalDevice(name='/device:GPU:0', device_type='GPU')]


I0000 00:00:1750182853.881740   42237 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9706 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


# Chapter 11: Deep Learning Text

Supporting materials for:

Chollet (2021). *Deep Learning with Python*. 2nd ed. Manning Publications Co.
[Amazon](https://www.amazon.com/Learning-Python-Second-Fran%C3%A7ois-Chollet/dp/1617296864/ref=sr_1_1?crid=32NFM2SBCJVQQ)

In this section we look at the particular types of deep network architectures that work well when processing textual time series, as
well as other aspects specific to preparing and processing textual input.

## 11.3 Two Approaches for Representing Groups of Words: Sets and sequences

- Simplest approach: discard order and treat as unordered set: **bag-of-words models**
- Process strictly in order they appear, like steps in a timeseries **sequence models**
- Hybrid approach: **Transformer architecture** technically order-agnostic, yet injects word-postiion info into representations.

In this section we'll return to the IMDB movie reviews dataset.  We'll demonstrate each approach (bag-of-words and sequence models) on
this dataset and see how they do.

In [ ]:
base_dir = pathlib.Path("../data/aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"


As hinted at, we have a similar subdirectory structure, so we can use a similar utility method from keras for streaming text
datasets called `text_dataset_from_directory`.  Let's create three `Dataset` objects for training, validation and testing:

**Note**: good idea to verify you get 20,000 files from train, 5,000 from validation after split and full 25,000 still in test datasets here.

In [ ]:
batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    train_dir, batch_size=batch_size
)

val_ds = keras.utils.text_dataset_from_directory(
    val_dir, batch_size=batch_size
)

test_ds = keras.utils.text_dataset_from_directory(
    test_dir, batch_size=batch_size
)

These datasets yield inputs that are TensorFlow `tf.string` tensors, and targets are `int32` tensors encoding values of
"0" or "1".

In [ ]:
for inputs, targets in train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

The inputs have not been tokenized or vectorized yet, they are just long strings here.

Notice that the 0'th input is labeled 0.  I think that because `neg` subdirectory name comes pefore `pos` subdirectory name, that we get the usual
encoding of negative reviews to 0 and positivie to 1.  I suspect that the `keras.utils` to stream from directories have an option to map subdirectory
names to desired label, or at a minimum you could always rename your directorys like `0-neg`, `1-pos` so they end up in order you want the
target labels to be assigned.

In [4]:
# reload train and test datasets and our TextVectorization instance for this notebook
base_dir = pathlib.Path("../data/aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"

batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    train_dir, batch_size=batch_size
)

val_ds = keras.utils.text_dataset_from_directory(
    val_dir, batch_size=batch_size
)

test_ds = keras.utils.text_dataset_from_directory(
    test_dir, batch_size=batch_size
)

# prepare a dataset that only yields raw text inputs (no labels)
text_only_train_ds = train_ds.map(lambda x, y: x)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


### 11.3.3 Processing words as a sequence

These past few examples clearly show that word order matters: manual engineering of
order-based features, such as bigrams, yields a nice accuracy boost.

What if, instead of manually crafting order-based features, we exposed the model to raw word sequences
and let it figure out such features on its own? This is what **sequence models** are about.

To implement a sequence model, you’d start by representing your input samples as
sequences of integer indices (one integer standing for one word). Then, you’d map
each integer to a vector to obtain vector sequences. Finally, you’d feed these
sequences of vectors into a stack of layers that could cross-correlate features from adjacent
vectors, such as a 1D convnet, a RNN, or a Transformer.

Let's try a sequence model.  First we need a dataset that returns integer sequences 
(rather than a multi-hot encoded vector representation).

In [5]:
# we will create sequences, but will make the maximum sequence length of 600 tokens.
# this means longer reviews will get choped to first 600 words, and shorts ones will be
# filled with the mask index 0 token
max_length = 600

# but we will still use most frequent 20,000 words for the vocabulary
max_tokens = 20000

# we'll truncate inputs after firt 600 words, this is a reasonable
# choice since the average review is 233 words and only 5%
# of reviews are longer than 600
# also since we are using sequences, bigram doesn't make sense, so we
# go back to 1-gram vocabulary
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)

# build the vocabulary again
text_vectorization.adapt(text_only_train_ds)

# The dataset instances again on the sequence output
# if we looked, what output would you expect from these,
# should be (32, 600) shaped outputs for batch size 32
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

In [ ]:
# just to be clear what we have now
for inputs, targets in int_train_ds:
    print("inputs.shape:", inputs.shape)
    print("inputs.dtype:", inputs.dtype)
    print("targets.shape:", targets.shape)
    print("targets.dtype:", targets.dtype)
    print("inputs[0]:", inputs[0])
    print("targets[0]:", targets[0])
    break

Next let's make a model.  We still have to convert integer sequences to
vector sequences.  The simplest way is to one-hot encode the integers (each dimension
would represent one possible term in the vocabulary).  On top of these one-hot
vectors we'll add a simple bidirectional LSTM.

In [ ]:
# also to be clear, if you didn't follow about the one-hot encoding
# if you one hot encode a sample of 600 input integers using 20,000 dimension encoding you get:
tf.one_hot(inputs[0], depth=max_tokens)

In [ ]:
# textbook shows directly using tf.one_hot() instance, but API has changed.
# example of subclassing the keras layer class
class TFOneHotEncodeIntSequence(keras.Layer):
    def call(self, x):
        return tf.one_hot(x, depth=max_tokens)


# one input is a sequence of integers 
inputs = keras.Input(shape=(None,), dtype="int64")
# encode the integers into binary 20,000 dimensional vectors
embedded = TFOneHotEncodeIntSequence()(inputs)
# add a bidirectional LSTM layer
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
# normal binary classification output layer
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

Now let's try it out and train this sequence model.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-one-hot-bidir-lstm.keras", save_best_only=True)
]

history = model.fit(int_train_ds, 
                    validation_data=int_val_ds, 
                    epochs=1,
                    callbacks=callbacks)

model = keras.models.load_model("../models/ch11-one-hot-bidir-lstm.keras",
                                custom_objects={"TFOneHotEncodeIntSequence": TFOneHotEncodeIntSequence})
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

You will first find that this model trains much more slowly than the previous ones.  This is because
the inputs are quite large, each input sample is encoded as a matrix of size `(600, 20000)`
(600 words per sample, 20,000 possible words).  That's 12,000,000 floats for a single movie review.

But if you can get this to train, you will find it doesn't improve performance at all, and it is maybe a bit
worse than the 90% we saw we could get.  Clearly using one-hot encoding to turn words into vectors, which was
the simplest thing we could do, wasn't a great idea.  There's a better way: **word embeddings**.

#### Understanding word embeddings

The fundamental assumption is that the different tokens you’re encoding are all independent from each other: indeed, one-hot vectors are all orthogonal
to one another.

In the case of words that assumption is clearly wrong.  Words form a structured space, they share information with each other.
The words “movie” and “film” are interchangeable in most sentences, so the vector that represents
“movie” should not be orthogonal to the vector that represents “film”—they should be
the same vector, or close enough. The geometric relationship between two word vectors
should reflect the semantic relationship between these words. 

**Word embeddings** are vector representations of words that achieve exactly this: they
map human language into a structured geometric space.

Whereas the vectors obtained through one-hot encoding are binary, sparse (mostly
made of zeros), and very high-dimensional (the same dimensionality as the number of
words in the vocabulary), word embeddings are low-dimensional floating-point vectors
(that is, dense vectors, as opposed to sparse vectors);

Besides being dense representations, word embeddings are also structured representations,
and their structure is learned from data. Similar words get embedded in close
locations, and further, specific directions in the embedding space are meaningful.

Two ways to obtain word embeddings

- Learn word embeddings jointly with the main task you care about (such as document
  classification or sentiment prediction). In this setup, you start with random
  word vectors and then learn word vectors in the same way you learn the
  weights of a neural network.
- Load into your model word embeddings that were precomputed using a different
  machine learning task than the one you’re trying to solve. These are called
  pretrained word embeddings.

#### Learning word embeddings with the `Embedding` layer

What makes a good word-embedding space depends heavily on
your task: the perfect word-embedding space for an English-language movie-review
sentiment-analysis model may look different from the perfect embedding space for an
English-language legal-document classification model.

The importance of certain semantic relationships varies from task to task.

Thus it is reasoanable to **learn** a new embedding space for your task.

Can train a parallel model (using same optimizaiton techniques you are familiar with)
using the Keras `Embedding` layer.


In [6]:
# the embedding layer takes at least two arguments, the number of possible tokens and the
# dimensionality of the embedding space to create (here 256)
embedding_layer = layers.Embedding(input_dim=max_tokens, output_dim=256)

The Embedding layer is best understood as a dictionary that maps integer indices
(which stand for specific words) to dense vectors. It takes integers as input, looks up
these integers in an internal dictionary, and returns the associated vectors. It’s effectively
a dictionary lookup

Once fully trained, the embedding space will show a lot of structure—a kind of structure specialized for the specific problem
for which you’re training your model.

Let's build a model that includes an `Embedding` layer and benchmark it on our task:

In [7]:
# similar to before, but instead of an embeded one-hot encoding layer,
# use Embedding layer to learn a word embedding space
# and in fact input is same sequence of int tokens as before
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 256)      │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 64)             │        73,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,194,049 (19.81 MB)

 Trainable params: 5,194,049 (19.81 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-embeddings-bidir-lstm.keras", save_best_only=True)
]

model.fit(int_train_ds, 
          validation_data=int_val_ds, 
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model("../models/ch11-embeddings-bidir-lstm.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

Epoch 1/10


I0000 00:00:1750184306.387119   42345 cuda_dnn.cc:529] Loaded cuDNN version 90300


625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 90ms/step - accuracy: 0.6253 - loss: 0.6261 - val_accuracy: 0.7484 - val_loss: 0.5547
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 92ms/step - accuracy: 0.8261 - loss: 0.4278 - val_accuracy: 0.8182 - val_loss: 0.4191
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 90ms/step - accuracy: 0.8739 - loss: 0.3402 - val_accuracy: 0.8610 - val_loss: 0.3349
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 93ms/step - accuracy: 0.8950 - loss: 0.2864 - val_accuracy: 0.8350 - val_loss: 0.4046
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - accuracy: 0.9104 - loss: 0.2450 - val_accuracy: 0.8410 - val_loss: 0.3715
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 90ms/step - accuracy: 0.9261 - loss: 0.2080 - val_accuracy: 0.8684 - val_loss: 0.3407
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 60s 96ms/step - accuracy: 0.9407 - loss: 0.1766 - val_accuracy: 0.8682 - val_loss: 0.3343
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 90ms/step - accuracy: 0.9518 - loss: 0.1492 - val_accurac

It trains much faster than the one-hot model (since LSTM only has to process a
256 dimensional vector instead of 20,000 dimensional).  And its text
accuracy is comparable, you will again probably only usually get 87% accuracy again.

Part of the reason may be the truncation to 600 words (according to text).

#### Understanding padding and masking

One thing that’s slightly hurting model performance here is that our input sequences
are full of zeros because most reviews are shorter than 600 words, so they are filled in with the MASK.

Sentences longer than 600 are truncated, and shorter are padded with 0's.

Recall how RNN work.  The RNN that looks at tokens in forward/natural order will spend its
last iterations a lot of times seing only 0's.  The information stored in the internal state
will gradually fade out as it gets exposed to these meaningless inputs.

We need some way to tell the RNN that it should skip these iterations. There’s an
API for that: **masking**.

The Embedding layer is capable of generating a “mask” that corresponds to its
input data. This mask is a tensor of ones and zeros (or True/False booleans), of shape
(batch_size, sequence_length), where the entry mask[i, t] indicates where timestep
t of sample i should be skipped or not (the timestep will be skipped if mask[i, t]
is 0 or False, and processed otherwise).

By default this option is off, you can turn it on by using `mazk_zero=True`
to your `Embedding` layer.

In [9]:
# example on made up input of what the embedding mask looks like
embedding_layer = layers.Embedding(input_dim=10, output_dim=256, mask_zero=True)

# made up input
some_input = [
    [4, 3, 2, 1, 0, 0, 0],
    [5, 4, 3, 2, 1, 0, 0],
    [2, 1, 0, 0, 0, 0, 0]]

mask = embedding_layer.compute_mask(some_input)
mask

<tf.Tensor: shape=(3, 7), dtype=bool, numpy=
array([[ True,  True,  True,  True, False, False, False],
       [ True,  True,  True,  True,  True, False, False],
       [ True,  True, False, False, False, False, False]])>

In practice you don't have to manage masks by hand.  Instead can automatically pass on the
mask to every layer that is able to process it.  This mask will be used by RNN layers
to skip masked steps.  

Let's retrain the previous mask with masking enabled this time.

In [10]:
# same as before, but try out using mask_zero=True in our word Embedding layer
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256, mask_zero=True)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 256) │  5,120,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer_1[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 64)        │     73,984 │ embedding_3[0][0… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 64)        │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         65 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,194,049 (19.81 MB)

 Trainable params: 5,194,049 (19.81 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-embeddings-bidir-lstm-with-masking.keras", save_best_only=True)
]

model.fit(int_train_ds, validation_data=int_val_ds, epochs=10, callbacks=callbacks)

model = keras.models.load_model("../models/ch11-embeddings-bidir-lstm-with-masking.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 93ms/step - accuracy: 0.6789 - loss: 0.5693 - val_accuracy: 0.8208 - val_loss: 0.3893
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 89ms/step - accuracy: 0.8635 - loss: 0.3297 - val_accuracy: 0.8520 - val_loss: 0.3383
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 90ms/step - accuracy: 0.8992 - loss: 0.2544 - val_accuracy: 0.7974 - val_loss: 0.4749
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 89ms/step - accuracy: 0.9209 - loss: 0.2111 - val_accuracy: 0.8684 - val_loss: 0.3254
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - accuracy: 0.9395 - loss: 0.1647 - val_accuracy: 0.8680 - val_loss: 0.3264
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 91ms/step - accuracy: 0.9574 - loss: 0.1205 - val_accuracy: 0.8740 - val_loss: 0.3690
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 92ms/step - accuracy: 0.9722 - loss: 0.0862 - val_accuracy: 0.8732 - val_loss: 0.3979
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 90ms/step - accuracy: 0.9788 - loss: 0.0656 - 

This time you should usually get a bit better than the most previous example without masking, though again still not better than
best model we have seen so far.

#### Using pretrained word embeddings

Sometimes you have so little training data available that you can’t use your data alone
to learn an appropriate task-specific embedding of your vocabulary.  In such cases,
it might be useful to use a pretrained embedding space (similar to how we used
pretrained models for computer vision previously).

To do this we have to download a pretrained embedding (these are not available directly
in the Keras library like the pretrained models we used before).

First let's download the GloVe word embeddings precomputed on the 2014 English
Wikipedia dataset.  It's an 822M zip file containing 100-dimensional embedding vectors for
400,000 words.

For the following examples I expect the files to be in the `../data` subdirectory.  Here is how to
get the embeddings in our class DevContainer environment:

**Note**: The certificate is expired on the url given here in the text.  Can use the `--no-check-certificate` flag to get,
though probably need to find if/where this data might be now.

```
vscode ➜ /workspaces/nndl/data (main) $ wget --no-check-certificate http://nlp.stanford.edu/data/glove.6B.zip

Saving to: ‘glove.6B.zip’
glove.6B.zip                       100%[=============================================================>] 822.24M  4.79MB/s    in 2m 49s  
2025-06-14 22:31:20 (4.87 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]


vscode ➜ /workspaces/nndl/data (main) $ unzip -q glove.6B.zip 
```

The result are several plain text files, named things like `glove.6B.100d.txt` where the `100d` indicates the number of embedding dimensions the
file contains.  These are rather big files (the `300d` file is about 1G unzipped).  We are going to use the `100d` 100 dimensions version,
so to save space I removed the zip file and all the other files:

```
vscode ➜ /workspaces/nndl/data (main) $ rm glove.6B.zip glove.6B.200d.txt glove.6B.300d.txt glove.6B.50d.txt 

```

The file is in a plain text format, so we need a little custom code to read it in.  We start by creating a dictionary that
maps each word to a set of coefficients (its embedding vector).

In [12]:
# location of the file to open and read in
path_to_glove_file = "../data/glove.6B.100d.txt"

# result of this first part is a simple dictionary that maps an english word
# to a key of the vector coefficients representing the vector encoding of this word.
embeddings_index = {}

with open(path_to_glove_file) as f:
    # each line of file first containes the word, followed by 100 coefficients
    for line in f:
        word, coefs = line.split(maxsplit=1)
        # this creates an numpy array, splitting by space separator
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs
print(f"Found {len(embeddings_index)} word vectors.")

Found 400000 word vectors.


In [13]:
# As an example, how is the word "movie" represented in this embedding.
print(embeddings_index['movie'])
print(type(embeddings_index['movie']))
print(embeddings_index['movie'].shape)

[ 0.3825  0.1482  0.606  -0.5153  0.4399  0.0611 -0.6272 -0.0254  0.1643
 -0.221   0.1442 -0.3721 -0.2168 -0.089   0.0979  0.6561  0.6446  0.477
  0.8385  1.6486  0.8892 -0.1181 -0.0125 -0.5208  0.7785  0.4872 -0.015
 -0.1413 -0.3475 -0.2959  0.1028  0.5719 -0.0456  0.0264  0.5382  0.3226
  0.4079 -0.0436 -0.146  -0.4835  0.3204  0.5509 -0.7626  0.4327  0.6175
 -0.365  -0.606  -0.7962  0.3929 -0.2367 -0.3472 -0.612   0.5475  0.9481
  0.2094 -2.7771 -0.6022  0.8495  1.2549  0.0179 -0.0419  2.1147 -0.0266
 -0.281   0.6812 -0.1417  0.9925  0.4988 -0.6754  0.6417  0.423  -0.2791
  0.0634  0.6891 -0.3618  0.0537 -0.1681  0.1942 -0.4707 -0.148  -0.5899
 -0.2797  0.1679  0.1057 -1.7601  0.0088 -0.8333 -0.5836 -0.3708 -0.5659
  0.207   0.0713  0.0556 -0.2976 -0.0727 -0.256   0.4269  0.0589  0.0911
  0.4728]
<class 'numpy.ndarray'>
(100,)


Next lets build an embedding matrix that you can load into an `Embedding` layer.
It must be a matrix of shape `(max_words, embedding_dim)` where each entry `i`
contains the `embedding_dim` dimensional vector for the word of index `i` in the
reference word index.

In [14]:
embedding_dim = 100

# retrieve the vocabulary indexed by our previous TextVectorization layer
# remember this is a list, orderd by the word index
vocabulary = text_vectorization.get_vocabulary()
# use it to create a mapping from words to their index in the vocabulary
# this is a dictionary with the word as key and its index as the value now
word_index = dict(zip(vocabulary, range(len(vocabulary))))

# the resulting embedding matrix has to have shape (max_tokens, embedding_dim)
# which in this particular case is (20000, 100) since we used 20,000 words and
# are reusing the 100 dimension word embedding
embedding_matrix = np.zeros((max_tokens, embedding_dim))

for word, i in word_index.items():
    # fill entry i in the matrix with word vector for index i
    # words not found in the embedding (not likely) will be all zeros
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

Finally we use a `Constant` initializer to load the pretrained embeddings in
an Embedding layer.  So as not to disrupt the pretrained representations during training,
we freeze the layer via `trainable=False`

In [15]:
embedding_layer = layers.Embedding(
    max_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True,
)  

We're now ready to train a new model, identical to our previous model, but leveraging the 100-dimensional pretrained GloVe embeddings instead of 256-dimensional
learned embeddings.

In [16]:
inputs = keras.Input(shape=(None,), dtype="int64")
embedded = embedding_layer(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer="rmsprop",
loss="binary_crossentropy",
metrics=["accuracy"])

model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, None, 100) │  2,000,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, None)      │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 64)        │     34,048 │ embedding_4[0][0… │
│ (Bidirectional)     │                   │            │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         65 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,034,113 (7.76 MB)

 Trainable params: 34,113 (133.25 KB)

 Non-trainable params: 2,000,000 (7.63 MB)

In [17]:
callbacks = [
    keras.callbacks.ModelCheckpoint("../models/ch11-glove-embeddings-sequence-model.keras", save_best_only=True)
]

model.fit(int_train_ds, 
          validation_data=int_val_ds,
          epochs=10,
          callbacks=callbacks)

model = keras.models.load_model("../models/ch11-glove-embeddings-sequence-model.keras")
print(f"Test acc: {model.evaluate(int_test_ds)[1]:.3f}")

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 58s 90ms/step - accuracy: 0.6286 - loss: 0.6320 - val_accuracy: 0.7266 - val_loss: 0.5308
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 89ms/step - accuracy: 0.7864 - loss: 0.4631 - val_accuracy: 0.8068 - val_loss: 0.4205
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 90ms/step - accuracy: 0.8149 - loss: 0.4194 - val_accuracy: 0.8316 - val_loss: 0.3897
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 89ms/step - accuracy: 0.8335 - loss: 0.3779 - val_accuracy: 0.8432 - val_loss: 0.3584
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 87ms/step - accuracy: 0.8500 - loss: 0.3506 - val_accuracy: 0.8242 - val_loss: 0.3872
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.8620 - loss: 0.3301 - val_accuracy: 0.8372 - val_loss: 0.3667
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 55s 88ms/step - accuracy: 0.8746 - loss: 0.3098 - val_accuracy: 0.8510 - val_loss: 0.3491
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 56s 89ms/step - accuracy: 0.8825 - loss: 0.2913 - 

You'll find that on this particular task, pretrained embeddings aren't very helpful because the dataset
contains enough samples that it is possible to learn a specialized enough embedding space from scratch.
However, leveraging pretrained embeddings can be very helpful when you're working with a smaller dataset.

## Summary

<font color='blue'>
    
- Word order in Text processing is handled in 2 basic ways:
  1. **bag-of-words models** : discard order and treat as an unordered set (multi-hot encoding typically, 20,000 sparse vectors).
  2. **sequence models**: process words in order they appear like a timeseries
- For sequence models, can encode sequence again using one-hot encoding, but this ends up with very large input to RNN models.
- **word embeddings** are vector representations of words that map human language into a structured geometric space.